In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Set up directory paths
notebook_path = Path().resolve()  # Path to the current notebook directory
experiments_path = notebook_path.parent # Path to the experiments directory where the utils module is located
backend_path = experiments_path.parent / "backend"

if experiments_path not in sys.path:
    sys.path.insert(0, str(experiments_path))

# Load environment variables from backend/.env if it exists;
# otherwise, load system environment variables with default behavior
env_path = backend_path / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"[OK] Loaded environment variables from {env_path}")
else:
    load_dotenv(override=True)
    print("[WARNING] Backend .env not found, using default environment")


In [ ]:
# --- Local Utilities ---
from utils.handbook_loader import load_handbook_documents

# Load all handbook documents from the backend data/handbook directory
all_documents = load_handbook_documents(backend_path / "data" / "handbook")[:5]

In [ ]:
from utils.prompts import CHUNK_GENERATION_SYSTEM_PROMPT
from utils.models import Chunk, Chunks, HandbookDoc

from litellm import completion
from tenacity import retry, wait_exponential
wait = wait_exponential(multiplier=1, min=10, max=240)

AVERAGE_CHUNK_SIZE = 400

@retry(wait=wait)
def generate_chunks_from_document(
    document: HandbookDoc,
    model: str = "groq/openai/gpt-oss-20b",
    max_tokens: int = 65536
) -> Chunks:
    """
    Generate chunks for a document.


    """
    number_chunks = (len(document.content) // AVERAGE_CHUNK_SIZE) + 1

    user_prompt = (
        f"The user has provided the following document from Made Tech's handbook:\n\n"
        f"[DOCUMENT METADATA]\n"
        f"id: {document.id}\n"
        f"title: {document.title}\n"
        f"category: {document.category}\n"
        f"[DOCUMENT METADATA END]\n\n"
        f"[DOCUMENTS BEGIN]\n\n{document.content}\n\n[DOCUMENTS END]\n\n\n"
        f"This document should probably be split into at least {number_chunks} chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions"
    )

    messages = [
        {"role": "system", "content": CHUNK_GENERATION_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    
    response = completion(
        model=model,
        messages=messages,
        response_format=Chunks,
        max_tokens=max_tokens
    )

    reply = response.choices[0].message.content
    
    # return Chunks.model_validate_json(reply)

    chunks = Chunks.model_validate_json(reply).chunks

    return [chunk.as_result(document) for chunk in chunks]

In [ ]:
from tqdm.notebook import tqdm
from multiprocessing import Pool
from typing import List

def sequential_chunk_generation(documents: List[HandbookDoc]):
    """
    Generate chunks for all documents sequentially, displaying a progress bar.
    """
    chunks = []
    for doc in tqdm(documents, desc="Generating chunks"):
        chunks += generate_chunks_from_document(doc)
    return chunks

chunks = sequential_chunk_generation(all_documents)


In [ ]:
len(chunks)

In [ ]:
from chromadb import PersistentClient
from openai import OpenAI
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
openai = OpenAI()

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
chunks[0].metadata

In [ ]:
create_embeddings(chunks)